# BCNexus — Stage-wise Workflow
**Author:** Md Eliasinul Islam

Run the CLEWs pipeline **one stage at a time**, via the Python API or the CLI — both drive the same `RunModel.stage_*` methods, so results are identical.

| Stage | Python | CLI |
|---|---|---|
| 1. Build SETs/params | `m.stage_build(include_livestock=INCLUDE_LIVESTOCK)` | `python -m bcnexus.stages build ...` |
| 2. Scenario overrides | `m.stage_scenario()` | `... scenario-overlay ...` |
| 3. otoole datafile | `m.stage_datafile()` | `... datafile ...` |
| 4. LP file (glpsol) | `m.stage_lp()` | `... lp ...` |
| 5. Solve (+duals/reports) | `m.stage_solve()` | `... solve ...` |
| 6. Result CSVs (otoole) | `m.stage_results()` | `... results ...` |

Why stages: after a crash (or a config tweak) you re-run only the affected step — e.g. edit `scenarios_bcnexus.yaml`, then re-run stages 2→6 without rebuilding.

> Run this notebook from the **repo root** (where `config/` and `data/` live).

## 0. Run definition — edit these

In [ ]:
SCENARIO      = "Base_CNZ_noCCS"   # key in config/scenarios_bcnexus.yaml
STORAGE_ALGO  = "Kotzur"           # Kotzur | Niet
HOUR_GROUPING = 4
N_CLUSTERS    = 1
SOLVER        = "gurobi"           # gurobi | cbc
THREADS       = 16

INCLUDE_LIVESTOCK = False  # True currently duplicates OAR rows (PWRBCWB01) -> GLPK "already defined"

---
# Option A — Python API
One `RunModel` instance; call stages in order. You can stop, inspect files, and continue — but note stages 3→5 share paths in memory, so within a *fresh* session start from the stage whose inputs already exist on disk (each stage re-derives what it needs).

In [ ]:
from bcnexus.clews.runner import RunModel

m = RunModel(
    run_scenario=SCENARIO,
    storage_algorithm=STORAGE_ALGO,
    clustering_attributes={"hour_grouping": HOUR_GROUPING, "n_clusters": N_CLUSTERS},
)

In [ ]:
# Stage 1 — CLEWs builder: SETs, params, temporal profiles, storage schema
built_csvs = m.stage_build(include_livestock=INCLUDE_LIVESTOCK)
built_csvs

In [ ]:
# Stage 2 — scenario overrides from config/scenarios_bcnexus.yaml
m.stage_scenario()

In [ ]:
# Stage 3 — otoole datafile + model file
data_file, model_file = m.stage_datafile()
print(data_file, model_file, sep="\n")

In [ ]:
# Stage 4 — LP file via glpsol
lp_file = m.stage_lp()
lp_file

In [ ]:
# Stage 5 — solve (writes .sol + duals, constraints summary, ELCB02 shadow price)
sol = m.stage_solve(solver_name=SOLVER, threads=THREADS)
sol   # None => not optimal; check the solver log in the run dir

In [ ]:
# Stage 6 — extract result CSVs via otoole
results_dir = m.stage_results(solver_name=SOLVER)
results_dir  # => .../{N}ts/result_csvs_gurobi/

Inspect any intermediate quickly:

In [ ]:
import pandas as pd
sorted(p.name for p in results_dir.glob("*.csv"))[:10] if results_dir else "no results yet"

---
# Option B — CLI
Same stages from the shell (what Snakemake calls). Each command is an **independent process** — safe to run days apart; every stage re-derives its paths from disk.

In [ ]:
ARGS = f"-s {SCENARIO} -a {STORAGE_ALGO} -hg {HOUR_GROUPING} -nc {N_CLUSTERS}"
print(ARGS)

LIVESTOCK_FLAG = "--include-livestock" if INCLUDE_LIVESTOCK else "--no-include-livestock"

In [ ]:
!python -m bcnexus.stages build {ARGS} {LIVESTOCK_FLAG}

In [ ]:
!python -m bcnexus.stages scenario-overlay {ARGS}

In [ ]:
!python -m bcnexus.stages datafile {ARGS}

In [ ]:
!python -m bcnexus.stages lp {ARGS}

In [ ]:
!python -m bcnexus.stages solve {ARGS} --solver {SOLVER} --threads {THREADS}

In [ ]:
!python -m bcnexus.stages results {ARGS} --solver {SOLVER}

---
# Option C — the full matrix (Snakemake)
Once single branches work, declare the scenario matrix in `workflow_v3/config.yaml` and let snakemake run all branches in parallel with resume:

In [ ]:
!snakemake -s workflow_v3/Snakefile -n     # dry run: audit what would run

In [ ]:
# !snakemake -s workflow_v3/Snakefile -c4  # uncomment to launch the matrix

## Notes
- Outputs land in the **deterministic** run dir `.../{N}ts/` (no more timestamped folders). Re-running a stage overwrites that branch only; run history/versioning is DVC's job (`dvc add`, see `workflow_v3/README_WORKFLOW.md`).
- `run()` (the classic one-shot) still works and is exactly stages 1→6 composed: `m.run(build=True, solver_name=SOLVER, threads=THREADS)`.
- Typical partial re-runs: scenario tweak → stages 2→6; timeslice/cluster change → stages 1→6; solver settings only → stages 5→6.